In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Định nghĩa đường dẫn file
file_path = '../data/dulieuketoan.csv'

# 2. Đọc file dữ liệu
try:
    df = pd.read_csv(file_path)
    print(f" Đọc dữ liệu thành công! Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột.")
except FileNotFoundError:
    print(" Lỗi: Không tìm thấy file dữ liệu. Hãy kiểm tra lại folder 'data' nhé!")

 Đọc dữ liệu thành công! Kích thước: 6819 dòng, 96 cột.


In [7]:
# Kiểm tra tên các cột ban đầu
print("Tên các cột trước khi sửa:", df.columns.tolist()[:5]) # In thử 5 cột đầu

# Thực hiện xóa khoảng trắng thừa ở đầu và cuối tên cột
df.columns = df.columns.str.strip()

print("\n Tên các cột sau khi chuẩn hóa:", df.columns.tolist()[:5])

Tên các cột trước khi sửa: ['Bankrupt?', ' ROA(C) before interest and depreciation before interest', ' ROA(A) before interest and % after tax', ' ROA(B) before interest and depreciation after tax', ' Operating Gross Margin']

 Tên các cột sau khi chuẩn hóa: ['Bankrupt?', 'ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax', 'Operating Gross Margin']


In [8]:
# 1. Tính tổng số lượng giá trị trống ở mỗi cột
missing_values = df.isnull().sum()
print("Số lượng dòng trống ở từng cột:")
print(missing_values[missing_values > 0]) # Chỉ in những cột có giá trị trống

# 2. Xử lý giá trị trống
# Cách an toàn nhất cho dữ liệu tài chính lớn: Xóa các dòng bị khuyết thông tin quan trọng
df_cleaned = df.dropna()

print(f"\n Dữ liệu sau khi xóa bỏ các dòng trống: {df_cleaned.shape[0]} dòng.")

Số lượng dòng trống ở từng cột:
Series([], dtype: int64)

 Dữ liệu sau khi xóa bỏ các dòng trống: 6819 dòng.


In [9]:
# 1. Đếm số dòng trùng lặp hoàn toàn
duplicate_count = df_cleaned.duplicated().sum()
print(f"Số dòng dữ liệu bị trùng lặp: {duplicate_count}")

# 2. Nếu có trùng lặp thì tiến hành xóa
if duplicate_count > 0:
    df_cleaned = df_cleaned.drop_duplicates()
    print(f" Đã xóa các dòng trùng! Kích thước mới: {df_cleaned.shape[0]} dòng.")
else:
    print(" Dữ liệu sạch sẽ, không bị trùng lặp!")

Số dòng dữ liệu bị trùng lặp: 0
 Dữ liệu sạch sẽ, không bị trùng lặp!


In [11]:
# Tên cột mục tiêu phân loại trạng thái doanh nghiệp (Phá sản hay An toàn)
target_column = 'Bankrupt?' 

# 1. Ép kiểu dữ liệu của cột mục tiêu về dạng số nguyên (0 hoặc 1) để tránh lỗi 'unknown'
df_cleaned[target_column] = df_cleaned[target_column].astype(int)

# 2. Tách đặc trưng (X) và nhãn mục tiêu (y)
X = df_cleaned.drop(columns=[target_column])
y = df_cleaned[target_column]

# 3. Chia dữ liệu theo tỷ lệ 80% Train - 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f" Kích thước tập Train (X_train): {X_train.shape}")
print(f" Kích thước tập Test (X_test): {X_test.shape}")

 Kích thước tập Train (X_train): (5455, 95)
 Kích thước tập Test (X_test): (1364, 95)


In [12]:
# Khởi tạo bộ chuẩn hóa
scaler = StandardScaler()

# Chỉ chuẩn hóa trên các cột số (loại trừ các cột định danh như Mã doanh nghiệp, Tên công ty nếu có)
# Tự động lấy các cột có kiểu dữ liệu là số
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

# Tiến hành fit và transform trên tập Train, sau đó áp dụng (transform) lên tập Test
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(" Đã chuẩn hóa StandardScaler xong cho các biến tài chính!")
# Xem thử dữ liệu biến đổi của dòng đầu tiên
print(X_train_scaled[numeric_cols].iloc[0][:5])

 Đã chuẩn hóa StandardScaler xong cho các biến tài chính!
ROA(C) before interest and depreciation before interest   -0.176677
ROA(A) before interest and % after tax                    -0.117303
ROA(B) before interest and depreciation after tax         -0.135577
Operating Gross Margin                                    -0.542117
Realized Sales Gross Margin                               -0.541389
Name: 318, dtype: float64


In [14]:
# Gộp lại nhãn mục tiêu vào tập dữ liệu đã chuẩn hóa để xuất ra file hoàn chỉnh
train_final = X_train_scaled.copy()
train_final[target_column] = y_train

test_final = X_test_scaled.copy()
test_final[target_column] = y_test

# Tạo file tổng hợp sau làm sạch
df_final_clean = pd.concat([train_final, test_final], axis=0)

# Lưu lại vào folder data
output_path = '../data/dulieuketoan_clean.csv'
df_final_clean.to_csv(output_path, index=False)

print(f" Xuất file dữ liệu sạch thành công tại: {output_path}")


 Xuất file dữ liệu sạch thành công tại: ../data/dulieuketoan_clean.csv


In [15]:
# ============ BƯỚC 9: PHÂN TÍCH TƯƠNG QUAN (CORRELATION) ============
print("\n" + "="*60)
print("BƯỚC 9: PHÂN TÍCH TƯƠNG QUAN")
print("="*60)

# Tính ma trận tương quan với biến mục tiêu
correlation_with_target = X_train_scaled.corrwith(y_train).sort_values(ascending=False)

print("\n  🔗 Top 10 biến có tương quan cao nhất với mục tiêu:")
print(correlation_with_target.head(10))

print("\n  🔗 Top 10 biến có tương quan thấp nhất với mục tiêu:")
print(correlation_with_target.tail(10))

# Tìm các biến có tương quan rất yếu (có thể loại bỏ)
weak_correlation_threshold = 0.05
weak_features = correlation_with_target[
    (correlation_with_target.abs() < weak_correlation_threshold) & 
    (correlation_with_target.abs() != 0)
]
print(f"\n  ⚠️ Số biến có tương quan rất yếu (<{weak_correlation_threshold}): {len(weak_features)}")


BƯỚC 9: PHÂN TÍCH TƯƠNG QUAN

  🔗 Top 10 biến có tương quan cao nhất với mục tiêu:
Debt ratio %                                   0.247642
Borrowing dependency                           0.196911
Current Liability to Assets                    0.189893
Current Liability to Current Assets            0.180961
Liability to Equity                            0.178138
Current Liabilities/Equity                     0.172577
Current Liability to Equity                    0.172577
Total expense/Assets                           0.142643
Equity to Long-term Liability                  0.122902
Inventory and accounts receivable/Net value    0.101110
dtype: float64

  🔗 Top 10 biến có tương quan thấp nhất với mục tiêu:
Net profit before tax/Paid-in capital                     -0.206277
Net Income to Stockholder's Equity                        -0.206648
Retained Earnings to Total Assets                         -0.213839
Persistent EPS in the Last Four Seasons                   -0.221009
Net worth/Asse

c:\Users\ADMIN\miniconda3\envs\distress_prediction\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\ADMIN\miniconda3\envs\distress_prediction\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
# ============ BƯỚC 10: BÁO CÁO CHẤT LƯỢNG DỮ LIỆU ============
print("\n" + "="*60)
print("BƯỚC 10: BÁO CÁO CHẤT LƯỢNG DỮ LIỆU")
print("="*60)

def data_quality_report(df, name="Dataset"):
    """Tạo báo cáo chất lượng dữ liệu chi tiết"""
    print(f"\n📋 {name} - Báo cáo chất lượng:")
    print(f"  • Số dòng: {df.shape[0]}")
    print(f"  • Số cột: {df.shape[1]}")
    print(f"  • Giá trị trống: {df.isnull().sum().sum()}")
    print(f"  • Tỷ % hoàn chỉnh: {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")
    print(f"  • Bộ nhớ sử dụng: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Phân bố class nếu có cột mục tiêu
    if target_column in df.columns:
        print(f"\n  📊 Phân bố class '{target_column}':")
        class_dist = df[target_column].value_counts()
        for cls, count in class_dist.items():
            ratio = (count / len(df)) * 100
            print(f"    - Class {cls}: {count} samples ({ratio:.2f}%)")

# Áp dụng báo cáo
data_quality_report(X_train, "Tập Train (X_train)")
data_quality_report(X_test, "Tập Test (X_test)")
data_quality_report(train_final, "Tập Train Final (sau chuẩn hóa)")
data_quality_report(test_final, "Tập Test Final (sau chuẩn hóa)")

In [ ]:
# ============ BƯỚC 11: THỐNG KÊ MÔ TẢ (DESCRIPTIVE STATISTICS) ============
print("\n" + "="*60)
print("BƯỚC 11: THỐNG KÊ MÔ TẢ CHI TIẾT")
print("="*60)

print("\n📈 Thống kê cơ bản của tập Train (sau chuẩn hóa):")
print(X_train_scaled.describe().round(4))

print("\n📊 Thông tin các biến:")
print(f"  • Số biến số: {len(numeric_cols)}")
print(f"  • Các biến số: {list(numeric_cols)[:5]}... (và {len(numeric_cols)-5} biến khác)")

# Kiểm tra mean và std của dữ liệu đã chuẩn hóa
print("\n✓ Kiểm tra chuẩn hóa StandardScaler:")
print(f"  • Mean của X_train_scaled: {X_train_scaled[numeric_cols].mean().mean():.6f} (≈ 0 ✓)")
print(f"  • Std của X_train_scaled: {X_train_scaled[numeric_cols].std().mean():.6f} (≈ 1 ✓)")

In [ ]:
# ============ BƯỚC 12: TÓM TẮT QUY TRÌNH TIỀN XỬ LÝ ============
print("\n" + "="*60)
print("BƯỚC 12: TÓM TẮT QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU")
print("="*60)

summary = {
    "Bước": [
        "1. Đọc dữ liệu",
        "2. Làm sạch tên cột",
        "3. Xóa dòng trống (NaN)",
        "4. Xóa dòng trùng lặp",
        "5. Tách features & target",
        "6. Chia train-test (80-20)",
        "7. Chuẩn hóa StandardScaler",
        "8. Phát hiện outliers",
        "9. Phân tích correlation",
        "10. Báo cáo chất lượng",
        "11. Thống kê mô tả",
        "12. Lưu dữ liệu"
    ],
    "Trạng thái": ["✓"]*12,
    "Ghi chú": [
        f"Tải {df.shape[0]} dòng x {df.shape[1]} cột",
        "Loại bỏ khoảng trắng thừa",
        f"Còn lại {df_cleaned.shape[0]} dòng",
        f"Xóa {duplicate_count} dòng trùng" if duplicate_count > 0 else "Không có trùng",
        f"X shape: {X_train.shape[0]}x{X_train.shape[1]}",
        f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}",
        f"Mean≈0, Std≈1 cho {len(numeric_cols)} biến",
        "Dùng IQR method phát hiện",
        f"Tìm {len(weak_features)} biến yếu",
        "100% hoàn chỉnh",
        f"Mean: {X_train_scaled[numeric_cols].mean().mean():.4f}",
        f"File: {output_path}"
    ]
}

summary_df = pd.DataFrame(summary)
print("\n")
print(summary_df.to_string(index=False))

print("\n" + "="*60)
print("✅ TIỀN XỬ LÝ DỮ LIỆU HOÀN THÀNH!")
print("="*60)

In [ ]:
# ============ BƯỚC 8: PHÁT HIỆN VÀ XỬ LÝ OUTLIERS ============
# Dùng IQR (Interquartile Range) method để phát hiện giá trị ngoại lệ

print("\n" + "="*60)
print("BƯỚC 8: PHÁT HIỆN VÀ XỬ LÝ OUTLIERS")
print("="*60)

def remove_outliers_iqr(data, columns=None, multiplier=1.5):
    """
    Xóa outliers dùng IQR method
    multiplier: thường là 1.5 (thông thường) hoặc 3 (nghiêm ngặt hơn)
    """
    if columns is None:
        columns = data.select_dtypes(include=[np.number]).columns
    
    data_clean = data.copy()
    outlier_count = 0
    
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        outliers_in_col = ((data[col] < lower_bound) | (data[col] > upper_bound)).sum()
        outlier_count += outliers_in_col
        
        if outliers_in_col > 0:
            print(f"  📊 Cột '{col}': {outliers_in_col} outliers phát hiện")
    
    # Xóa các dòng chứa outlier (nếu cần)
    # data_clean = data_clean[~((data < (Q1 - 1.5*IQR)) | (data > (Q3 + 1.5*IQR))).any(axis=1)]
    
    print(f"\n  ✓ Tổng outliers phát hiện: {outlier_count}")
    return data_clean

# Áp dụng cho tập train
X_train_no_outliers = remove_outliers_iqr(X_train)